# M3KG-RAG reproduction (scaled-down, AudioCaps-QA) — Colab driver
Runs `scripts/m3kg/pipeline.sh`. Default profile: `configs/m3kg/profiles/colab_l4.yaml`.

### 0. Session setup
Runtime → *Change runtime type* → **L4 GPU** (or A100). Store `WANDB_API_KEY` and `HF_TOKEN` in Colab *Secrets* (key icon on the left).

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
for k in ('WANDB_API_KEY', 'HF_TOKEN'):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print('secret not set:', k)
os.environ['WANDB_ENTITY'] = 'YOUR_WANDB_ENTITY'   # <- edit
os.environ['ENV_ROOT'] = '/content/envs'

In [ ]:
%cd /content
!git clone https://github.com/YOUR_GITHUB_USER/mmgraphrag-repro.git 2>/dev/null || (cd mmgraphrag-repro && git pull)
%cd /content/mmgraphrag-repro
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

### 1. Environments (once per VM)

In [ ]:
!bash scripts/setup_envs.sh vllm m3kg-audio

### 2. Data: AudioCaps-QA test set + AudioCaps train subset (corpus)

In [ ]:
!bash scripts/colab/sync.sh pull
!bash scripts/m3kg/pipeline.sh data && bash scripts/colab/sync.sh push

### 3. (optional) YouTube metadata for the rewriter agent
Then add `SET='kg.rewriter=true'` to the next cell.

In [ ]:
!bash scripts/m3kg/pipeline.sh youtube

### 4. Multi-agent KG construction (Qwen3 + Wikipedia/Wiktionary)

In [ ]:
!WIKI_USER_AGENT='mmgraphrag-repro/0.1 (YOUR_EMAIL)' bash scripts/m3kg/pipeline.sh kg && bash scripts/colab/sync.sh push

### 5. CLAP index, retrieval + TAG grounding for all configurations

In [ ]:
!bash scripts/m3kg/pipeline.sh index contexts && bash scripts/colab/sync.sh push

### 6. GRASP LLM filter → Qwen2.5-Omni generation → AudioBench judge

In [ ]:
!bash scripts/m3kg/pipeline.sh filter generate judge && bash scripts/colab/sync.sh push

### 7. (optional) pairwise win-rate judge, then metrics → W&B

In [ ]:
!bash scripts/m3kg/pipeline.sh winrate report && bash scripts/colab/sync.sh push

### 8. W&B report pages + LaTeX tables

In [ ]:
!/content/envs/m3kg-audio/bin/python scripts/make_wandb_reports.py --share
!/content/envs/m3kg-audio/bin/python scripts/export_results.py --results results --out results/latex